> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Basic INNER JOIN


In [ ]:
-- INNER JOIN drops payments without a registered customer; filter to successful payments.
SELECT t.transaction_id, c.country, c.customer_name, t.amount, t.status
FROM payment_transactions t
    INNER JOIN customers c ON t.customer_id = c.customer_id
WHERE
    t.status = 'Success'
ORDER BY t.transaction_id DESC;

## INNER JOIN with aggregation

In [ ]:
-- Group matched successful payments by customer to calculate count, total and average.
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(transaction_id) AS total_transactions,
    SUM(t.amount) AS total_amount,
    AVG(t.amount) AS average_amount
FROM customers c
    INNER JOIN payment_transactions t ON c.customer_id = t.customer_id
WHERE
    t.status = 'Success'
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_amount DESC;

## LEFT JOIN

In [ ]:
-- LEFT JOIN preserves customers without payments as rows with NULL payment fields.
SELECT c.customer_id, c.customer_name, t.transaction_id, t.amount, t.status
FROM customers c
    LEFT JOIN payment_transactions t ON c.customer_id = t.customer_id
ORDER BY c.customer_id, t.transaction_id;

## LEFT JOIN with NULL filtering

In [ ]:
-- A NULL payment ID after LEFT JOIN identifies customers with no payments.
SELECT c.customer_id, c.customer_name, c.country
FROM customers c
    LEFT JOIN payment_transactions t ON c.customer_id = t.customer_id
WHERE
    t.transaction_id IS NULL;

## LEFT JOIN with conditional aggregation

In [ ]:
-- Count payment IDs rather than COUNT(*) so customers without payments have a count of zero.
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(t.transaction_id) AS total_transactions,
    SUM(
        CASE
            WHEN t.status = 'Success' THEN 1
            ELSE 0
        END
    ) AS successful_transactions,
    SUM(
        CASE
            WHEN t.status = 'Failed' THEN 1
            ELSE 0
        END
    ) AS failed_transactions,
    SUM(
        CASE
            WHEN t.status = 'Success' THEN t.amount
            ELSE 0
        END
    ) AS total_successful_amount,
    COALESCE(SUM(t.amount), 0) AS total_amount
FROM customers c
    LEFT JOIN payment_transactions t ON c.customer_id = t.customer_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_amount DESC, c.customer_id ASC;

## RIGHT JOIN


In [ ]:
-- RIGHT JOIN preserves every payment, including the orphan customer ID 999.
SELECT
t.transaction_id,
t.amount,
t.status,
c.customer_id,
c.customer_name
FROM customers c
RIGHT JOIN payment_transactions t ON c.customer_id = t.customer_id
ORDER BY t.transaction_id DESC;

## SELF JOIN

In [ ]:
-- Join customers to themselves to retrieve referrer names; only referred customers remain.
SELECT c.customer_id,
c.customer_name,
r.customer_name AS referred_by_name
FROM customers c
INNER JOIN customers r
    on c.referred_by = r.customer_id
ORDER BY c.customer_id;

## SELF JOIN using LEFT JOIN

In [ ]:
-- Preserve all customers and label missing referrers for display.
SELECT c.customer_id, c.customer_name, COALESCE(
        r.customer_name, 'No Referrer'
    ) AS referred_by_name
FROM customers c
    LEFT JOIN customers r ON c.referred_by = r.customer_id
ORDER BY c.customer_id;